In [1]:
import torch
import torchvision.transforms.functional as F
import numpy as np
from sklearn.datasets import fetch_lfw_people
from torch.utils.data import Dataset, DataLoader
import random

In [2]:
print("Downloading dataset via scikit-learn...")
lfw = fetch_lfw_people(min_faces_per_person = 5, color = True, resize=1.0, slice_=None)
images = lfw.images
labels = lfw.target
names = lfw.target_names
print(f"Successfully loaded {len(images)} images!")
print(f"Raw Numpy Shape: {images.shape} (N, H, W, C)")

Successfully loaded 5985 images!
Raw Numpy Shape: (5985, 250, 250, 3) (N, H, W, C)


In [3]:
images_transposed = np.transpose(images, (0, 3, 1, 2))

tensor_images = torch.tensor(images_transposed).float()

tensor_images = F.resize(tensor_images, size=[128, 128])

print(f"Final PyTorch Tensor Shape: {tensor_images.shape}")


Final PyTorch Tensor Shape: torch.Size([5985, 3, 128, 128])


In [5]:
class TripleFaceDataset(Dataset):
    def __init__(self, tensor_images, labels):
        self.images = tensor_images
        self.labels = labels
        self.unique_labels = np.unique(labels)
        # Label mapped with respective indices
        self.label_to_indices = {label: np.where(self.labels == label)[0] for label in self.unique_labels}
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, index):
        anchor_img = self.images[index]
        anchor_label = self.labels[index]

        positive_idx = random.choice(self.label_to_indices[anchor_label])
        positive_img = self.images[positive_idx]

        negative_label = random.choice(labels)
        while(negative_label == anchor_label):
            negative_label = random.choice(labels)
        
        negative_idx = random.choice(self.label_to_indices[negative_label])
        negative_img = self.images[negative_idx]

        return anchor_img, positive_img, negative_img



In [7]:
dataset = TripleFaceDataset(tensor_images, labels)

dataloader = DataLoader(dataset, batch_size=32, shuffle=True, drop_last=True)

print(f"Total Batches ready for training: {len(dataloader)}")

anchor_batch, positive_batch, negative_batch = next(iter(dataloader))
print(f"One Anchor Batch Shape: {anchor_batch.shape}")

Total Batches ready for training: 187
One Anchor Batch Shape: torch.Size([32, 3, 128, 128])
